# Aqua Milk Detect — SVM trainer (notebook)

Same pipeline as `train.py`, cell by cell, so you can poke at the data. Both share
`utils.py`, so they cannot drift apart.

1. Load and clean the collected CSVs
2. Build features (order fixed by `libs/AquaMilkSensors/src/features.h`)
3. Check balance — the warnings matter more than the accuracy
4. Grid-search linear vs RBF with stratified k-fold, macro-F1
5. Evaluate honestly on a held-out split
6. Sweep the Uncertain threshold
7. Verify the firmware's probability maths, then export `model.h` + `scaler.h`

Set `SYNTHETIC = True` to run end-to-end before you have real data.

In [ ]:
import numpy as np, pathlib, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix)
import utils

SYNTHETIC  = False      # True = generated data, for a smoke test
DATA_GLOB  = "data/*.csv"
CHAMBER_ML = 100.0      # must match the value stored on the device
SEED       = 42
HERE = pathlib.Path.cwd()

In [ ]:
df = utils.synthetic_frame(seed=SEED) if SYNTHETIC else utils.load_csvs(DATA_GLOB)
df = utils.clean(df)
warnings = utils.check_balance(df)
for w in warnings:
    print("WARNING:", w)
df.head()

In [ ]:
X = utils.raw_to_features(df, CHAMBER_ML)
y = df[utils.LABEL].to_numpy()
print("feature order:", list(X.columns))
X.describe().T

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
scaler = StandardScaler().fit(X_tr)
Xs_tr, Xs_te = scaler.transform(X_tr), scaler.transform(X_te)

_, inv = np.unique(y_tr, return_inverse=True)
folds = int(min(5, np.bincount(inv).min()))
grid = [
    {"kernel": ["linear"], "C": [0.1, 1, 10, 100]},
    {"kernel": ["rbf"], "C": [1, 10, 100, 1000], "gamma": ["scale", 0.01, 0.1, 1]},
]
search = GridSearchCV(SVC(probability=True, random_state=SEED), grid, scoring="f1_macro",
                      cv=StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEED),
                      n_jobs=-1).fit(Xs_tr, y_tr)
clf = search.best_estimator_
print(search.best_params_, f"CV macro-F1 {search.best_score_:.3f} over {folds} folds")

In [ ]:
y_hat = clf.predict(Xs_te)
print(f"accuracy {accuracy_score(y_te, y_hat):.3f}   macro-F1 {f1_score(y_te, y_hat, average='macro'):.3f}")
print(classification_report(y_te, y_hat, zero_division=0))

labels = list(clf.classes_)
cm = confusion_matrix(y_te, y_hat, labels=labels)
fig, ax = plt.subplots(figsize=(4.4, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels)), labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i, j], ha="center", va="center")
ax.set_xlabel("predicted"); ax.set_ylabel("truth")
plt.show()

bin_true = np.where(y_te == "pure", "pure", "adulterated")
bin_pred = np.where(y_hat == "pure", "pure", "adulterated")
print(f"binary pure-vs-adulterated accuracy {accuracy_score(bin_true, bin_pred):.3f}")
print("Starch is normally the weakest class — it moves turbidity the way fat does.")

In [ ]:
proba = clf.predict_proba(Xs_te)
top, pred = proba.max(axis=1), clf.classes_[proba.argmax(axis=1)]
ths = np.arange(0.40, 0.91, 0.05)
cov = [(top >= t).mean() for t in ths]
acc = [accuracy_score(y_te[top >= t], pred[top >= t]) if (top >= t).any() else np.nan for t in ths]

plt.plot(ths, cov, "-o", label="coverage")
plt.plot(ths, acc, "-s", label="accuracy when answered")
plt.xlabel("confidence threshold"); plt.ylim(0, 1.05); plt.legend(); plt.grid(alpha=.3)
plt.show()

ok = [(t, a, c) for t, a, c in zip(ths, acc, cov) if not np.isnan(a) and c >= 0.7]
THRESHOLD = round(float(max(ok, key=lambda r: (r[1], r[2]))[0]), 2) if ok else 0.60
print(f"recommended threshold {THRESHOLD:.2f} → "
      f"{int((top < THRESHOLD).sum())} of {len(y_te)} test rows become Uncertain")

In [ ]:
# Proves the ESP32's probability path (svm_infer.h) reproduces predict_proba.
# It raises rather than exporting a model whose confidences the device would get wrong.
utils.verify_probabilities(clf, scaler.transform(X))

utils.export_scaler_h(HERE / "scaler.h", scaler.mean_, scaler.scale_, THRESHOLD)
utils.export_model_h(HERE / "model.h", clf, Xs_tr)
print("\nCopy model.h and scaler.h into 03_deployment/ and reflash.")

## Before you quote any number above

- A few dozen rows per class means accuracy moves several points on a reshuffle. Report
  the confusion matrix.
- The model learnt *your* device's raw millivolts under *your* calibration. Recalibrating
  or swapping a probe invalidates it.
- Only the adulteration levels you actually prepared are in scope.
- `train.py` writes all of this — plus permutation importance and a limitations section —
  into `report/report.txt`, which is the version to hand in.